# Cognitive Stress Model - Draft

Analyzing physiological signals from 22 subjects across two protocols.

**Subjects:**
- **V1 (males with Stroop):** S04, S05, S08, S09, S10, S13, S14, S15, S17, S18
- **V2 (females without Stroop):** f01, f02, f03, f04, f05, f06, f08, f09, f10, f11, f12, f13

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## Configuration

In [ ]:
# Paths
dataset_path = '22subjects/STRESS'
stress_level_v1_path = 'WISE_data_files/Stress_Level_v1.csv'
stress_level_v2_path = 'WISE_data_files/Stress_Level_v2.csv'

# Subject lists
v1_subjects = ['S04', 'S05', 'S08', 'S09', 'S10', 'S13', 'S14', 'S15', 'S17', 'S18']
v2_subjects = ['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13']
all_subjects = v1_subjects + v2_subjects

print(f"V1 Subjects (n={len(v1_subjects)}): {v1_subjects}")
print(f"V2 Subjects (n={len(v2_subjects)}): {v2_subjects}")
print(f"Total: {len(all_subjects)} subjects")

## Load Self-Reported Stress Levels

In [ ]:
# Load stress level data
stress_level_v1 = pd.read_csv(stress_level_v1_path, index_col=0)
stress_level_v2 = pd.read_csv(stress_level_v2_path, index_col=0)

print("V1 Stress Levels (Males with Stroop):")
print(f"Phases: {list(stress_level_v1.columns)}")
display(stress_level_v1)

print("\nV2 Stress Levels (Females without Stroop):")
print(f"Phases: {list(stress_level_v2.columns)}")
display(stress_level_v2)

## Helper Functions

In [ ]:
def create_df_array(dataframe):
    """Converts a pandas DataFrame to a flattened numpy array."""
    return dataframe.values.flatten()


def time_abs_(UTC_array):
    """Converts UTC timestamps to seconds from the start of recording."""
    new_array = []
    start_time = datetime.datetime.strptime(UTC_array[0], '%Y-%m-%d %H:%M:%S')
    
    for utc in UTC_array:
        current_time = datetime.datetime.strptime(utc, '%Y-%m-%d %H:%M:%S')
        seconds_elapsed = (current_time - start_time).total_seconds()
        new_array.append(int(seconds_elapsed))
    
    return new_array


def moving_average(acc_data):
    """
    Applies a moving average filter to accelerometer data to measure movement.
    Higher values = more movement, Lower values = less movement
    """
    avg = 0
    prevX, prevY, prevZ = 0, 0, 0
    results = []
    
    for i in range(0, len(acc_data), 32):
        sum_ = 0
        buffX = acc_data[i:i+32, 0]
        buffY = acc_data[i:i+32, 1]
        buffZ = acc_data[i:i+32, 2]
        
        for j in range(len(buffX)):
            sum_ += max(
                abs(buffX[j] - prevX),
                abs(buffY[j] - prevY),
                abs(buffZ[j] - prevZ)
            )
            prevX, prevY, prevZ = buffX[j], buffY[j], buffZ[j]
        
        avg = avg * 0.9 + (sum_ / 32) * 0.1
        results.append(avg)
    
    return results

print("Helper functions defined")

## Load Physiological Signals

In [ ]:
def read_signals(main_folder):
    """
    Read all physiological signals from subject folders.
    Each subject folder contains: EDA, BVP, HR, IBI, TEMP, ACC, tags
    """
    signal_dict = {}
    time_dict = {}
    fs_dict = {}
    
    subfolders = next(os.walk(main_folder))[1]
    
    # Get start times
    utc_start_dict = {}
    for folder_name in subfolders:
        csv_path = f'{main_folder}/{folder_name}/EDA.csv'
        df = pd.read_csv(csv_path)
        utc_start_dict[folder_name] = df.columns.tolist()
    
    # Read all signals
    for folder_name in subfolders:
        folder_path = os.path.join(main_folder, folder_name)
        files = os.listdir(folder_path)
        
        signals = {}
        time_line = {}
        fs_signal = {}
        
        desired_files = ['EDA.csv', 'BVP.csv', 'HR.csv', 'TEMP.csv', 'tags.csv', 'ACC.csv', 'IBI.csv']
        
        for file_name in files:
            if file_name not in desired_files:
                continue
            
            file_path = os.path.join(folder_path, file_name)
            signal_name = file_name.replace('.csv', '')
            
            if file_name == 'tags.csv':
                try:
                    df = pd.read_csv(file_path, header=None)
                    tags_vector = create_df_array(df)
                    tags_UTC_vector = np.insert(tags_vector, 0, utc_start_dict[folder_name])
                    signal_array = time_abs_(tags_UTC_vector)
                except pd.errors.EmptyDataError:
                    signal_array = []
            
            elif file_name == 'IBI.csv':
                df = pd.read_csv(file_path)
                signal_array = df.values
                fs_signal['IBI'] = 'variable'
            
            else:
                df = pd.read_csv(file_path)
                fs = int(df.iloc[0, 0])
                signal_array = df.iloc[1:].values
                time_array = np.linspace(0, len(signal_array)/fs, len(signal_array))
                
                time_line[signal_name] = time_array
                fs_signal[signal_name] = fs
            
            signals[signal_name] = signal_array
        
        signal_dict[folder_name] = signals
        time_dict[folder_name] = time_line
        fs_dict[folder_name] = fs_signal
    
    return signal_dict, time_dict, fs_dict

# Load all signals
print("Loading physiological signals...")
signal_data, time_data, fs_dict = read_signals(dataset_path)

# Verify we have all 22 subjects
loaded_subjects = list(signal_data.keys())
print(f"\nLoaded {len(loaded_subjects)} subjects: {sorted(loaded_subjects)}")

## Define Protocol Phases

In [ ]:
# Phase colors for consistent visualization
PHASE_COLORS = {
    # STRESS phases (red/orange tones)
    'Stroop': '#e74c3c',
    'TMCT': '#e67e22',
    'Real Opinion': '#f39c12',
    'Opposite Opinion': '#f1c40f',
    'Subtract': '#d35400'
}

# Signal colors
SIGNAL_COLORS = {
    'EDA': '#2ecc71',
    'BVP': '#e74c3c',
    'HR': '#3498db',
    'TEMP': '#9b59b6',
    'ACC': '#e67e22'
}

def get_stress_rest_segments(subject_id, tags):
    """
    Get protocol segments with phase labels.
    V1: Baseline(R) -> Stroop(S) -> Rest(R) -> TMCT(S) -> Rest(R) -> Speeches(S) -> Subtract(S)
    V2: Baseline(R) -> TMCT(S) -> Rest(R) -> Speeches(S) -> Rest(R) -> Subtract(S)
    """
    segments = []
    
    if subject_id.startswith('S'):  # V1 protocol
        if len(tags) >= 13:
            segments.append({'start': tags[0], 'end': tags[3], 'label': 'REST', 'phase': 'Baseline'})
            segments.append({'start': tags[3], 'end': tags[4], 'label': 'STRESS', 'phase': 'Stroop'})
            segments.append({'start': tags[4], 'end': tags[5], 'label': 'REST', 'phase': 'First Rest'})
            segments.append({'start': tags[5], 'end': tags[6], 'label': 'STRESS', 'phase': 'TMCT'})
            segments.append({'start': tags[6], 'end': tags[7], 'label': 'REST', 'phase': 'Second Rest'})
            segments.append({'start': tags[7], 'end': tags[8], 'label': 'STRESS', 'phase': 'Real Opinion'})
            segments.append({'start': tags[8], 'end': tags[9], 'label': 'REST', 'phase': 'Transition Rest 1'})
            segments.append({'start': tags[9], 'end': tags[10], 'label': 'STRESS', 'phase': 'Opposite Opinion'})
            segments.append({'start': tags[10], 'end': tags[11], 'label': 'REST', 'phase': 'Transition Rest 2'})
            segments.append({'start': tags[11], 'end': tags[12], 'label': 'STRESS', 'phase': 'Subtract'})
    
    else:  # V2 protocol
        if len(tags) >= 10:
            segments.append({'start': tags[0], 'end': tags[2], 'label': 'REST', 'phase': 'Baseline'})
            segments.append({'start': tags[2], 'end': tags[3], 'label': 'STRESS', 'phase': 'TMCT'})
            segments.append({'start': tags[3], 'end': tags[4], 'label': 'REST', 'phase': 'First Rest'})
            segments.append({'start': tags[4], 'end': tags[5], 'label': 'STRESS', 'phase': 'Real Opinion'})
            segments.append({'start': tags[5], 'end': tags[6], 'label': 'REST', 'phase': 'Transition Rest'})
            segments.append({'start': tags[6], 'end': tags[7], 'label': 'STRESS', 'phase': 'Opposite Opinion'})
            segments.append({'start': tags[7], 'end': tags[8], 'label': 'REST', 'phase': 'Second Rest'})
            segments.append({'start': tags[8], 'end': tags[9], 'label': 'STRESS', 'phase': 'Subtract'})
    
    return segments

print("Phase definitions loaded")

## Plotting Function with Legends

In [ ]:
def plot_subject_signals_with_legend(subject_id, signals, time_dict, tags):
    """
    Plot all physiological signals for one subject with phase legends.
    """
    # Determine protocol type
    protocol = "V1 (Male, with Stroop)" if subject_id.startswith('S') else "V2 (Female, without Stroop)"
    
    # Get segments for this subject
    segments = get_stress_rest_segments(subject_id, tags)
    
    # Create figure
    fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True)
    fig.suptitle(f'{subject_id} - {protocol}\nPhysiological Signals Across Protocol Phases', 
                 fontsize=14, fontweight='bold', y=1.02)
    
    signal_names = ['EDA', 'BVP', 'HR', 'TEMP', 'ACC']
    signal_labels = [
        'EDA (Electrodermal Activity)',
        'BVP (Blood Volume Pulse)',
        'HR (Heart Rate)',
        'TEMP (Temperature)',
        'ACC (Accelerometer)'
    ]
    
    # Track which phases we've added to legend (to avoid duplicates)
    phase_handles = {}
    
    for idx, (ax, signal_name, signal_label) in enumerate(zip(axes, signal_names, signal_labels)):
        
        # Plot signal
        if signal_name in signals and signal_name in time_dict:
            if signal_name == 'ACC':
                acc_filtered = moving_average(signals[signal_name])
                time_acc = np.linspace(0, len(signals[signal_name])/32, len(acc_filtered))
                ax.plot(time_acc, acc_filtered, color=SIGNAL_COLORS[signal_name], 
                       linewidth=0.8, label=signal_label)
            else:
                ax.plot(time_dict[signal_name], signals[signal_name], 
                       color=SIGNAL_COLORS[signal_name], linewidth=0.5, label=signal_label)
        
        # Add phase shading (only for STRESS phases that have colors)
        for seg in segments:
            phase = seg['phase']
            if phase in PHASE_COLORS:  # Only shade STRESS phases
                color = PHASE_COLORS[phase]
                alpha = 0.3
                patch = ax.axvspan(seg['start'], seg['end'], color=color, alpha=alpha)
                
                # Store handle for legend (only once per phase)
                if phase not in phase_handles:
                    phase_handles[phase] = patch
        
        # Add tag markers (vertical dashed lines)
        for tag in tags[1:]:
            ax.axvline(x=tag, color='gray', linestyle='--', alpha=0.5, linewidth=0.5)
        
        # Formatting
        ax.set_ylabel(signal_name, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis='both', labelsize=9)
    
    # X-axis label on bottom plot
    axes[-1].set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    
    # Create custom legend for STRESS phases only
    from matplotlib.patches import Patch
    legend_elements = []
    
    legend_elements.append(Patch(facecolor='none', edgecolor='none', label='STRESS Phases:'))
    for phase, handle in phase_handles.items():
        legend_elements.append(Patch(facecolor=PHASE_COLORS[phase], alpha=0.5, label=f'  {phase}'))
    
    # Place legend outside the plot
    fig.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.01, 0.5),
               fontsize=9, framealpha=0.95, title='Protocol Phases', title_fontsize=10)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.85)
    plt.show()
    
    return segments

print("Plotting function defined")

---
## Plot All 22 Subjects

### V1 Subjects (Males with Stroop Test)

In [ ]:
# Plot V1 subjects
print("="*60)
print("V1 SUBJECTS (Males with Stroop Test)")
print("="*60)

for subject_id in v1_subjects:
    if subject_id in signal_data:
        print(f"\n--- {subject_id} ---")
        segments = plot_subject_signals_with_legend(
            subject_id,
            signal_data[subject_id],
            time_data[subject_id],
            signal_data[subject_id]['tags']
        )
        
        # Print segment summary
        print(f"Segments for {subject_id}:")
        for seg in segments:
            dur = seg['end'] - seg['start']
            print(f"  {seg['label']:7s} | {seg['phase']:20s} | {seg['start']:4d}-{seg['end']:4d}s ({dur:3d}s)")
    else:
        print(f"WARNING: {subject_id} not found in data!")

### V2 Subjects (Females without Stroop Test)

In [ ]:
# Plot V2 subjects
print("="*60)
print("V2 SUBJECTS (Females without Stroop Test)")
print("="*60)

for subject_id in v2_subjects:
    if subject_id in signal_data:
        print(f"\n--- {subject_id} ---")
        segments = plot_subject_signals_with_legend(
            subject_id,
            signal_data[subject_id],
            time_data[subject_id],
            signal_data[subject_id]['tags']
        )
        
        # Print segment summary
        print(f"Segments for {subject_id}:")
        for seg in segments:
            dur = seg['end'] - seg['start']
            print(f"  {seg['label']:7s} | {seg['phase']:20s} | {seg['start']:4d}-{seg['end']:4d}s ({dur:3d}s)")
    else:
        print(f"WARNING: {subject_id} not found in data!")

---
## Summary: Self-Reported Stress by Phase

In [ ]:
# Plot mean stress levels by phase for both protocols
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# V1
mean_v1 = stress_level_v1.mean()
colors_v1 = [PHASE_COLORS.get(phase, '#999999') for phase in mean_v1.index]
mean_v1.plot(kind='bar', ax=axes[0], color=colors_v1, edgecolor='black', alpha=0.8)
axes[0].set_title('V1: Mean Self-Reported Stress by Phase\n(Males with Stroop)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Stress Level (0-10)', fontsize=11)
axes[0].set_xlabel('Phase', fontsize=11)
axes[0].axhline(y=5, color='red', linestyle='--', alpha=0.5, label='Moderate Stress (5)')
axes[0].set_ylim(0, 8)
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

# V2
mean_v2 = stress_level_v2.mean()
colors_v2 = [PHASE_COLORS.get(phase, '#999999') for phase in mean_v2.index]
mean_v2.plot(kind='bar', ax=axes[1], color=colors_v2, edgecolor='black', alpha=0.8)
axes[1].set_title('V2: Mean Self-Reported Stress by Phase\n(Females without Stroop)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Stress Level (0-10)', fontsize=11)
axes[1].set_xlabel('Phase', fontsize=11)
axes[1].axhline(y=5, color='red', linestyle='--', alpha=0.5, label='Moderate Stress (5)')
axes[1].set_ylim(0, 8)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary
print("\nV1 Protocol - Mean Stress by Phase:")
for phase, stress in mean_v1.sort_values(ascending=False).items():
    print(f"  {phase:25s}: {stress:.2f}")

print("\nV2 Protocol - Mean Stress by Phase:")
for phase, stress in mean_v2.sort_values(ascending=False).items():
    print(f"  {phase:25s}: {stress:.2f}")

---
## Lasso Regression Model

Lasso (Least Absolute Shrinkage and Selection Operator) regression is useful for:
- **Feature Selection**: Automatically shrinks unimportant feature coefficients to zero
- **Interpretability**: Shows which features matter most for predicting stress
- **Regularization**: Prevents overfitting with L1 penalty

In [ ]:
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix
from scipy.signal import butter, filtfilt, find_peaks

print("Lasso imports loaded")

### Feature Extraction

In [ ]:
def calculate_hrv(bvp_data, fs_bvp, start_time, end_time):
    """
    Calculate HRV from BVP (Blood Volume Pulse) data.
    Returns overall_var and beat_to_beat_var.
    """
    if len(bvp_data) == 0:
        return {'overall_var': np.nan, 'beat_to_beat_var': np.nan}
    
    idx_start = int(start_time * fs_bvp)
    idx_end = int(end_time * fs_bvp)
    bvp_segment = bvp_data[idx_start:idx_end].flatten()
    
    if len(bvp_segment) < fs_bvp * 5:
        return {'overall_var': np.nan, 'beat_to_beat_var': np.nan}
    
    # Bandpass filter
    nyquist = fs_bvp / 2
    low = 0.5 / nyquist
    high = min(4.0 / nyquist, 0.99)
    b, a = butter(2, [low, high], btype='band')
    bvp_filtered = filtfilt(b, a, bvp_segment)
    
    # Detect peaks
    min_distance = int(fs_bvp * 0.3)
    peaks, _ = find_peaks(bvp_filtered, distance=min_distance, height=0)
    
    if len(peaks) < 3:
        return {'overall_var': np.nan, 'beat_to_beat_var': np.nan}
    
    # Calculate RR intervals
    rr_intervals = np.diff(peaks) / fs_bvp * 1000
    valid_rr = rr_intervals[(rr_intervals > 300) & (rr_intervals < 2000)]
    
    if len(valid_rr) < 2:
        return {'overall_var': np.nan, 'beat_to_beat_var': np.nan}
    
    overall_var = np.std(valid_rr)
    successive_diffs = np.diff(valid_rr)
    beat_to_beat_var = np.sqrt(np.mean(successive_diffs ** 2))
    
    return {'overall_var': overall_var, 'beat_to_beat_var': beat_to_beat_var}


def extract_segment_features(subject, segment, signals, fs_dict):
    """Extract all biomarkers for one segment"""
    features = {
        'subject': subject,
        'label': segment['label'],
        'phase': segment['phase'],
        'duration': segment['end'] - segment['start']
    }
    
    start_t, end_t = segment['start'], segment['end']
    
    # HRV from BVP
    if 'BVP' in signals and len(signals['BVP']) > 0:
        fs_bvp = fs_dict[subject]['BVP']
        hrv = calculate_hrv(signals['BVP'], fs_bvp, start_t, end_t)
        features.update(hrv)
    else:
        features.update({'overall_var': np.nan, 'beat_to_beat_var': np.nan})
    
    # EDA
    if 'EDA' in signals:
        fs = fs_dict[subject]['EDA']
        idx_start, idx_end = int(start_t * fs), int(end_t * fs)
        eda = signals['EDA'][idx_start:idx_end].flatten()
        features['EDA_mean'] = np.mean(eda)
        features['EDA_std'] = np.std(eda)
        features['EDA_max'] = np.max(eda)
    
    # HR
    if 'HR' in signals:
        fs = fs_dict[subject]['HR']
        idx_start, idx_end = int(start_t * fs), int(end_t * fs)
        hr = signals['HR'][idx_start:idx_end].flatten()
        features['HR_mean'] = np.mean(hr)
        features['HR_std'] = np.std(hr)
        features['HR_max'] = np.max(hr)
    
    # Temperature
    if 'TEMP' in signals:
        fs = fs_dict[subject]['TEMP']
        idx_start, idx_end = int(start_t * fs), int(end_t * fs)
        temp = signals['TEMP'][idx_start:idx_end].flatten()
        features['TEMP_mean'] = np.mean(temp)
        features['TEMP_std'] = np.std(temp)
    
    # Accelerometer
    if 'ACC' in signals:
        fs = fs_dict[subject]['ACC']
        idx_start, idx_end = int(start_t * fs), int(end_t * fs)
        acc = signals['ACC'][idx_start:idx_end]
        acc_filtered = moving_average(acc)
        features['ACC_mean'] = np.mean(acc_filtered) if len(acc_filtered) > 0 else np.nan
        features['ACC_std'] = np.std(acc_filtered) if len(acc_filtered) > 0 else np.nan
    
    return features

print("Feature extraction functions defined")

In [ ]:
# Build segments for all subjects
all_segments = {}
for subject in all_subjects:
    if subject in signal_data:
        tags = signal_data[subject]['tags']
        if len(tags) > 0:
            segments = get_stress_rest_segments(subject, tags)
            all_segments[subject] = segments

# Extract features for all segments
all_features = []
for subject in all_subjects:
    if subject in all_segments:
        for segment in all_segments[subject]:
            feat = extract_segment_features(subject, segment, signal_data[subject], fs_dict)
            all_features.append(feat)

df_features = pd.DataFrame(all_features)

print(f"Extracted {len(df_features)} segments")
print(f"  STRESS: {len(df_features[df_features['label']=='STRESS'])}")
print(f"  REST: {len(df_features[df_features['label']=='REST'])}")
print(f"\nFeature columns: {list(df_features.columns)}")
df_features.head()

### Lasso Regression for Classification

Using Lasso to predict stress label (0=REST, 1=STRESS) and identify important features.

In [ ]:
# Define feature columns (biosignal features only)
feature_cols = ['overall_var', 'beat_to_beat_var', 'EDA_mean', 'EDA_std', 
                'HR_mean', 'HR_std', 'TEMP_mean', 'TEMP_std', 
                'ACC_mean', 'ACC_std']

# Prepare data
X = df_features[feature_cols]
y = (df_features['label'] == 'STRESS').astype(int)  # 0=REST, 1=STRESS

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Features: {feature_cols}")

In [ ]:
# Step 1: Preprocess data (impute missing values and scale)
imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imputed)

X_test_imputed = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imputed)

# Step 2: Use LassoCV to find optimal alpha (regularization strength)
# Higher alpha = more regularization = more coefficients pushed to zero
alphas = np.logspace(-4, 1, 50)  # Range of alphas to try

lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)

print(f"Optimal alpha: {lasso_cv.alpha_:.6f}")
print(f"Number of non-zero coefficients: {np.sum(lasso_cv.coef_ != 0)}/{len(feature_cols)}")

In [ ]:
# Step 3: Analyze Lasso Coefficients (Feature Importance)
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lasso_cv.coef_,
    'abs_coef': np.abs(lasso_cv.coef_)
}).sort_values('abs_coef', ascending=False)

print("="*60)
print("LASSO COEFFICIENTS (Feature Importance)")
print("="*60)
print("Positive = associated with STRESS")
print("Negative = associated with REST")
print("Zero = feature eliminated by Lasso")
print("="*60)

for _, row in coef_df.iterrows():
    status = "SELECTED" if row['coefficient'] != 0 else "eliminated"
    print(f"{row['feature']:20s}: {row['coefficient']:+.4f}  ({status})")

# Visualize coefficients
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#d62728' if c > 0 else '#1f77b4' if c < 0 else '#cccccc' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Lasso Coefficient', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.set_title('Lasso Regression Coefficients\n(Red = STRESS, Blue = REST, Gray = Eliminated)', 
             fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Evaluate Model Performance
y_pred_continuous = lasso_cv.predict(X_test_scaled)
y_pred = (y_pred_continuous >= 0.5).astype(int)  # Threshold at 0.5 for classification

print("="*60)
print("MODEL EVALUATION")
print("="*60)

# Regression metrics
print(f"\nRegression Metrics:")
print(f"  R² Score: {r2_score(y_test, y_pred_continuous):.3f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_continuous)):.3f}")

# Classification metrics (using 0.5 threshold)
print(f"\nClassification Metrics (threshold=0.5):")
print(classification_report(y_test, y_pred, target_names=['REST', 'STRESS']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

### Lasso Regularization Path

Visualize how coefficients change as regularization strength (alpha) increases.

In [ ]:
# Compute Lasso path (coefficients for different alphas)
alphas_path = np.logspace(-4, 0, 100)
coefs = []

for alpha in alphas_path:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    coefs.append(lasso.coef_)

coefs = np.array(coefs)

# Plot regularization path
fig, ax = plt.subplots(figsize=(12, 6))

for i, feature in enumerate(feature_cols):
    ax.plot(alphas_path, coefs[:, i], label=feature, linewidth=2)

ax.axvline(x=lasso_cv.alpha_, color='black', linestyle='--', linewidth=2, label=f'Optimal α={lasso_cv.alpha_:.4f}')
ax.set_xscale('log')
ax.set_xlabel('Alpha (Regularization Strength)', fontsize=12)
ax.set_ylabel('Coefficient Value', fontsize=12)
ax.set_title('Lasso Regularization Path\n(How coefficients shrink as alpha increases)', 
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Features that stay non-zero longer (as alpha increases) are more important")
print("- Features that quickly go to zero are less useful for prediction")

In [ ]:
# Summary: Selected Features by Lasso
selected_features = coef_df[coef_df['coefficient'] != 0]['feature'].tolist()

print("="*60)
print("LASSO FEATURE SELECTION SUMMARY")
print("="*60)
print(f"\nFeatures SELECTED by Lasso ({len(selected_features)}/{len(feature_cols)}):")
for feat in selected_features:
    coef = coef_df[coef_df['feature'] == feat]['coefficient'].values[0]
    direction = "→ STRESS" if coef > 0 else "→ REST"
    print(f"  • {feat}: {coef:+.4f} {direction}")

eliminated = coef_df[coef_df['coefficient'] == 0]['feature'].tolist()
if eliminated:
    print(f"\nFeatures ELIMINATED by Lasso ({len(eliminated)}):")
    for feat in eliminated:
        print(f"  ✗ {feat}")

print(f"\n→ Use these {len(selected_features)} features for your final model!")